# Extract XLM-R hidden states on Colab T4

Run this notebook top-to-bottom. Estimated total time on a T4: **10–20 minutes** for all 5 languages at target_per_class=3000.

**Before you start:** enable a GPU runtime (Runtime → Change runtime type → T4 GPU).

## 1. Confirm GPU is available

In [ ]:
!nvidia-smi | head -20

## 2. Clone your repo and install deps

Replace `<YOUR-USERNAME>` with your GitHub handle.

In [ ]:
!git clone https://github.com/<YOUR-USERNAME>/xlmr-number-probing.git
%cd xlmr-number-probing
!pip install -q -r requirements.txt

## 3. Sanity check: subword alignment on one row

Before extracting hidden states for thousands of rows, verify that
`word_idx` correctly points to the target noun after subword tokenization.
If the printed subword tokens don't look like the target noun, stop and
check that you rerun the updated `extract_ud.py`.

In [ ]:
import pandas as pd
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained('xlm-roberta-base')

for lang in ['english', 'spanish', 'arabic', 'hindi', 'finnish']:
    df = pd.read_csv(f'data/balanced/{lang}_train.csv')
    row = df.iloc[0]
    tokens = row['tokens'].split()
    enc = tok([tokens], is_split_into_words=True)
    word_ids = enc.word_ids(0)
    subword_positions = [i for i, w in enumerate(word_ids) if w == row['word_idx']]
    subwords = [enc.tokens(0)[i] for i in subword_positions]
    print(f'{lang:8s} form={row["form"]!r:15s} subwords={subwords}')

**What to expect:** the subwords should visibly correspond to the target
noun. For English 'individuals' you might see `['▁individuals']` or
`['▁individual', 's']`. For Arabic or Hindi you'll see script-appropriate
subwords. If any language shows subwords totally unrelated to the form,
there's an alignment bug.

## 4. Run the extraction

In [ ]:
!python src/extract_hidden_states.py \
    --input-dir data/balanced \
    --output-dir results/hidden_states \
    --batch-size 32

## 5. Quick post-extraction check

In [ ]:
import torch
from pathlib import Path

for pt in sorted(Path('results/hidden_states').glob('*.pt')):
    d = torch.load(pt, map_location='cpu')
    h = d['hidden_states']
    y = d['labels']
    print(f'{pt.name:25s} hidden={tuple(h.shape)} dtype={h.dtype} labels={y.bincount().tolist()}')

Expected: 15 files (5 langs × 3 splits), shapes like `(2100, 13, 768)` for
train and `(600, 13, 768)` for test, labels perfectly balanced (or almost
— a few dropped from truncation is fine).

## 6. Zip and download

Zips to ~500MB. Download to your laptop for Week 3.

In [ ]:
!zip -r hidden_states.zip results/hidden_states
!ls -lh hidden_states.zip

from google.colab import files
files.download('hidden_states.zip')